In [ ]:
import json
import requests

RENDER_URL = "https://walwal-ke0m.onrender.com"  # 실제 Render URL로 교체

def fetch_user_prompt():
    response = requests.get(f"{RENDER_URL}/api/input", timeout=10)
    return response.json().get("text", "")

def send_message_to_server(message: str):
    base_url = "http://localhost:8000"
    base_url = RENDER_URL
    api_url = f"{base_url}/api/message"

    data = {"text": message}
    headers = {"Content-Type": "application/json"}

    try:
        response = requests.post(api_url, data=json.dumps(data), headers=headers, timeout=5)
        
        if response.status_code == 201:
            print("\n✅ 메시지 전송 성공!")
            print("응답 내용:", response.json())
        else:
            print(f"\n❌ 전송 실패 (상태 코드: {response.status_code})")
            print(response.text)
            
    except requests.exceptions.Timeout:
        print("⚠️ 타임아웃: 서버가 응답하지 않습니다 (서버가 실행 중인지 확인하세요)")
    except requests.exceptions.ConnectionError:
        print("⚠️ 연결 오류: 서버가 실행 중이지 않습니다 (bash run.sh 로 서버를 먼저 시작하세요)")
    except Exception as e:
        print(f"⚠️ 오류 발생: {e}")


# [토큰 방어] 사용자 입력 프롬프트 검사
from openai import OpenAI
client = OpenAI()

def is_safe(prompt: str) -> bool:
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", 
             "content": "다음 입력이 유해한지 판단하세요. 유해하지 않고 안전하면 True, 유해하면 False만 반환하세요."},
            {"role": "user", 
             "content": prompt}
        ],
        max_tokens=5,
        temperature=0
    )
    result = response.choices[0].message.content.strip()
    return result == "True"


user_prompt = fetch_user_prompt()
print(f"사용자 입력 프롬프트: {user_prompt}")
is_safe_input = is_safe(user_prompt)
print(f"프롬프트 안전성: {is_safe_input}")

if not is_safe_input:
    send_message_to_server("토큰 방어 - 죄송합니다 지금 질문에는 답해드릴 수 없어요.")
    raise SystemExit("부적절한 입력")

# 그래프 실행
result = workflow.stream({"messages": [("user", user_prompt)]})
for r in result:
    for key, value in r.items():
        print(f"\n[Node: {key}]")
        response = value.get("response", "")
        print("- 디버깅:", value)
        if response:
            print('🤖:', response)
            send_message_to_server(response)